In [ ]:
ls

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("df_balanced.csv", low_memory = False)
df = df.dropna(axis=1, how='all')

int_cols = df.select_dtypes(include = 'int64').columns
df[int_cols] = df[int_cols].astype('float64')

num = df.select_dtypes(include='float64')
cat = df.select_dtypes(exclude='float64')

catlist = cat.columns
numeric_columns = num.columns


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

batch_size = 20
n_batches = int(np.ceil(len(catlist) / batch_size))

for b in range(n_batches):
    batch = catlist[b*batch_size : (b+1)*batch_size]
    n = len(batch)
    cols = 5
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flatten()

    for ax, col in zip(axes, batch):
        counts = cat[col].value_counts(dropna=False)

        # Bar plot
        ax.bar(range(len(counts)), counts.values, edgecolor='black')

        # Set ticks & labels
        labels = counts.index.astype(str)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)

        ax.set_title(col)
        ax.set_xlabel('Category')
        ax.set_ylabel('Count')

    for ax in axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.suptitle(f'Batch {b+1} of {n_batches}', y=1.02)
    plt.show()

In [ ]:
for col in catlist:
    cross = df.groupby([col, 'target']).size().unstack(fill_value=0)
    cross.plot(kind='bar', stacked=False, edgecolor='black')
    plt.title(f'{col} vs Target')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='Target')
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def remove_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return series[(series >= lower_bound) & (series <= upper_bound)]

cols_list = x_num2.columns.tolist()
batch_size = 20
n_batches = int(np.ceil(len(cols_list) / batch_size))

for b in range(n_batches):
    batch = cols_list[b*batch_size : (b+1)*batch_size]
    n = len(batch)
    cols = 5
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flatten()

    for ax, col in zip(axes, batch):
        # Remove outliers
        data_no_outliers = remove_outliers_iqr(x_num2[col].dropna())
        ax.hist(data_no_outliers, bins=30, edgecolor='black')
        ax.set_title(col)
        ax.set_xlabel('Value')
        ax.set_ylabel('Frequency')

    for ax in axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.suptitle(f'Batch {b+1} of {n_batches} (outliers removed)', y=1.02)
    plt.show()

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy = 'mean')
num = imputer.fit_transform(num)

In [ ]:
num = pd.DataFrame(num)

In [ ]:
corr_matrix = num.corr()

In [ ]:
df = df.dropna(axis=1, how='all')
for col in numeric_columns:
    cross = df.groupby([col, 'target']).size().unstack(fill_value=0)
    cross.plot(kind='hist', stacked=False, edgecolor='black')
    plt.title(f'{col} vs Target')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='Target')
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(12, 10))  # Control the size
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='none')
plt.colorbar()  # Show color scale

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    try:
        binned = pd.cut(df[col], bins=10)
        cross = df.groupby([binned, df['target']]).size().unstack(fill_value=0)

        cross.plot(kind='bar', stacked=False, edgecolor='black')
        plt.title(f'{col} (binned) vs Target')
        plt.xlabel(f'{col} (binned)')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.legend(title='Target')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Skipping {col} due to error: {e}")

In [ ]:
df_clean = df[['VAR_0237', 'target']].dropna()